In [1]:
import pandas as pd
import numpy as np

close = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/학습학습/부산_월별폐업_쇼핑_식음료_핵심학습용.csv',
    dtype={'연월': str}
)

biz = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/학습학습/2022-2026_사업자_현황_6대업종_통합.csv',
    dtype={'연월': str}
)

# 부산 전체 합계 제외 → 16개 구군만
biz = biz[
    (biz['지역'] != '부산광역시 합계') &
    (biz['업종'].isin(['쇼핑업', '식음료업']))
].copy()

# 숫자형 변환
num_cols = ['당월①', '전월②', '전년동월③']

for col in num_cols:
    biz[col] = pd.to_numeric(biz[col], errors='coerce')

# 우리가 사용할 변수명으로 정리
biz['사업자수'] = biz['당월①']
biz['전월사업자수'] = biz['전월②']

biz['사업자_증감수'] = (
    biz['당월①'] - biz['전월②']
)

biz['사업자_MoM'] = (
    (biz['당월①'] - biz['전월②'])
    / biz['전월②'] * 100
)

biz['사업자_YoY'] = (
    (biz['당월①'] - biz['전년동월③'])
    / biz['전년동월③'] * 100
)

# 폐업 + 사업자 현황 결합
df = close.merge(
    biz,
    left_on=['연월', '지역', '6대업종'],
    right_on=['연월', '지역', '업종'],
    how='inner'
)

df['date'] = pd.to_datetime(
    df['연월'],
    format='%Y%m'
)

print(df.shape)
df.head()

(544, 23)


,연월,지역,원업종,6대업종,폐업수,6대업종매핑신뢰도,지역합계검증,원천발행연월,원천값유형,중복검증원천수,...,전월②,증감율(①/②),전년동월③,증감율(①/③),사업자수,전월사업자수,사업자_증감수,사업자_MoM,사업자_YoY,date
0,202410,강서구,소매업,쇼핑업,63,중간,통과,202510,전년동월,1,...,3737,100.32,3703,101.24,3749,3737,12,0.321113,1.242236,2024-10-01
1,202410,강서구,음식점업,식음료업,37,높음,통과,202510,전년동월,1,...,2680,99.93,2737,97.84,2678,2680,-2,-0.074627,-2.155645,2024-10-01
2,202410,금정구,소매업,쇼핑업,60,중간,통과,202510,전년동월,1,...,4385,99.84,4451,98.36,4378,4385,-7,-0.159635,-1.640081,2024-10-01
3,202410,금정구,음식점업,식음료업,46,높음,통과,202510,전년동월,1,...,3387,99.88,3491,96.91,3383,3387,-4,-0.118099,-3.093669,2024-10-01
4,202410,기장군,소매업,쇼핑업,62,중간,통과,202510,전년동월,1,...,4049,100.20,4014,101.07,4057,4049,8,0.197580,1.071251,2024-10-01


In [2]:
# 폐업 시차 변수 만들기.
# 실제 달력 기준 lag 생성 필요
def add_closure_lag(data, lag):
    lag_df = data[
        ['date', '지역', '6대업종', '폐업수']
    ].copy()

    # 과거값을 lag개월 미래 날짜로 이동
    lag_df['date'] = (
        lag_df['date']
        + pd.DateOffset(months=lag)
    )

    lag_df = lag_df.rename(
        columns={
            '폐업수': f'폐업_lag{lag}'
        }
    )

    return data.merge(
        lag_df,
        on=['date', '지역', '6대업종'],
        how='left'
    )

for lag in [1, 2, 3]:
    df = add_closure_lag(df, lag)

# 월 계절성
month = df['date'].dt.month

df['month_sin'] = np.sin(
    2 * np.pi * month / 12
)

df['month_cos'] = np.cos(
    2 * np.pi * month / 12
)

In [3]:
# 학습 데이터 선정 - 3개월치 과거 데이터가 모두 존재하는 경우만 우선 사용해보기

model_df = df.dropna(
    subset=[
        '폐업_lag1',
        '폐업_lag2',
        '폐업_lag3',
        '사업자수',
        '사업자_MoM',
        '사업자_YoY'
    ]
).copy()

print(model_df['연월'].value_counts().sort_index())

연월
202501    32
202502    32
202503    32
202504    32
202505    32
202512    32
202601    32
202602    32
202603    32
202604    32
202605    32
Name: count, dtype: int64


In [4]:
# 랜덤 분할 방지
# 미래 데이터가 훈련 데이터 속으로 들어가 데이터 누수가 발생하는 일을 방지하기 위해
# 훈련은 3월까지의 데이터로,
train = model_df[
    model_df['date'] <= '2026-03-01'
].copy()

test = model_df[
    model_df['date'].between(
        '2026-04-01',
        '2026-05-01'
    )
].copy()

print("TRAIN :", train.shape)
print("TEST :", test.shape)

TRAIN : (288, 28)
TEST : (64, 28)


In [5]:
# Poisson Regression 모델로 분석 진행

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error

categorical_features = [
    '지역',
    '6대업종'
]

numeric_features = [
    '사업자수',
    '사업자_증감수',
    '사업자_MoM',
    '사업자_YoY',
    '폐업_lag1',
    '폐업_lag2',
    '폐업_lag3',
    'month_sin',
    'month_cos'
]

numeric_transformer = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='median')
    ),
    (
        'scaler',
        StandardScaler()
    )
])

categorical_transformer = Pipeline([
    (
        'onehot',
        OneHotEncoder(
            handle_unknown='ignore'
        )
    )
])

preprocessor = ColumnTransformer([
    (
        'num',
        numeric_transformer,
        numeric_features
    ),
    (
        'cat',
        categorical_transformer,
        categorical_features
    )
])

model = Pipeline([
    (
        'preprocessor',
        preprocessor
    ),
    (
        'model',
        PoissonRegressor(
            alpha=0.5,
            max_iter=2000
        )
    )
])

X_train = train[
    categorical_features
    + numeric_features
]

y_train = train['폐업수']

X_test = test[
    categorical_features
    + numeric_features
]

y_test = test['폐업수']

model.fit(
    X_train,
    y_train
)

pred = model.predict(X_test)

test['예측폐업수'] = pred

In [6]:
# Baseline 과 비교 필요
baseline = test['폐업_lag1']

model_mae = mean_absolute_error(
    y_test,
    pred
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline
)

print(
    "Poisson MAE :",
    model_mae
)

print(
    "지난달값 Baseline MAE :",
    baseline_mae
)

Poisson MAE : 8.952592143223491
지난달값 Baseline MAE : 9.78125


In [7]:
# 오차 확인

print("실제 폐업수 평균 :", y_test.mean())
print("실제 폐업수 중앙값 :", y_test.median())

print(
    "평균 대비 MAE 비율 :",
    model_mae / y_test.mean() * 100
)

실제 폐업수 평균 : 52.21875
실제 폐업수 중앙값 : 46.5
평균 대비 MAE 비율 : 17.144401471164077


In [8]:
# 지역별 / 업종별 성능 확인
test['절대오차'] = abs(
    test['폐업수'] - test['예측폐업수']
)

print(
    test.groupby('6대업종')['절대오차']
        .mean()
)

print(
    test.groupby('지역')['절대오차']
        .mean()
        .sort_values()
)

6대업종
쇼핑업     10.414701
식음료업     7.490484
Name: 절대오차, dtype: float64
지역
영도구      1.527620
연제구      2.489661
사상구      4.238734
기장군      4.513212
동구       4.560767
서구       5.166790
북구       6.286476
남구       6.437000
강서구      6.537176
중구       8.785881
금정구      9.710221
사하구     11.133930
동래구     11.213499
해운대구    11.277846
수영구     18.007626
부산진구    31.355034
Name: 절대오차, dtype: float64


In [9]:
# 예측값, 실제값 비교

result = test[
    ['연월', '지역', '6대업종', '폐업수']
].copy()

result['예측폐업수'] = test['예측폐업수']
result['오차'] = (
    result['예측폐업수']
    - result['폐업수']
)

result.head(20)

,연월,지역,6대업종,폐업수,예측폐업수,오차
480,202604,강서구,쇼핑업,36,44.145085,8.145085
481,202604,강서구,식음료업,28,32.979330,4.979330
482,202604,금정구,쇼핑업,49,63.548978,14.548978
483,202604,금정구,식음료업,39,44.451516,5.451516
484,202604,기장군,쇼핑업,58,56.652371,-1.347629
485,202604,기장군,식음료업,39,34.499385,-4.500615
486,202604,남구,쇼핑업,61,75.243205,14.243205
487,202604,남구,식음료업,50,44.109326,-5.890674
488,202604,동구,쇼핑업,36,39.775370,3.775370
489,202604,동구,식음료업,20,22.932837,2.932837


In [10]:
# 산점도로 판단
import plotly.express as px

fig = px.scatter(
    result,
    x='폐업수',
    y='예측폐업수',
    color='6대업종',
    hover_data=['지역', '연월'],
    title='실제 폐업수 vs 예측 폐업수'
)

max_val = max(
    result['폐업수'].max(),
    result['예측폐업수'].max()
)

fig.add_shape(
    type='line',
    x0=0,
    y0=0,
    x1=max_val,
    y1=max_val
)

fig.show()

In [11]:
region_eval = test.groupby('지역').agg(
    실제평균=('폐업수', 'mean'),
    MAE=('절대오차', 'mean')
)

region_eval['평균대비_MAE_pct'] = (
    region_eval['MAE']
    / region_eval['실제평균']
    * 100
)

region_eval.sort_values(
    '평균대비_MAE_pct'
)

,실제평균,MAE,평균대비_MAE_pct
지역,,,
연제구,52.75,2.489661,4.719736
영도구,20.25,1.527620,7.543801
기장군,49.25,4.513212,9.163883
사상구,46.00,4.238734,9.214639
남구,59.00,6.437000,10.910169
북구,56.25,6.286476,11.175957
해운대구,88.00,11.277846,12.815734
동구,27.75,4.560767,16.435198
동래구,66.00,11.213499,16.990150


In [12]:
future = biz[
    biz['연월'] == '202606'
].copy()

future['date'] = pd.to_datetime(
    future['연월'],
    format='%Y%m'
)

future = future.rename(
    columns={
        '업종': '6대업종'
    }
)

# 폐업 과거값 생성용
close_lag = close.copy()

close_lag['date'] = pd.to_datetime(
    close_lag['연월'],
    format='%Y%m'
)

for lag in [1, 2, 3]:

    lag_df = close_lag[
        [
            'date',
            '지역',
            '6대업종',
            '폐업수'
        ]
    ].copy()

    lag_df['date'] = (
        lag_df['date']
        + pd.DateOffset(months=lag)
    )

    lag_df = lag_df.rename(
        columns={
            '폐업수': f'폐업_lag{lag}'
        }
    )

    future = future.merge(
        lag_df,
        on=[
            'date',
            '지역',
            '6대업종'
        ],
        how='left'
    )

month = future['date'].dt.month

future['month_sin'] = np.sin(
    2 * np.pi * month / 12
)

future['month_cos'] = np.cos(
    2 * np.pi * month / 12
)

X_future = future[
    categorical_features
    + numeric_features
]

future['예측폐업수'] = model.predict(
    X_future
)

future[
    [
        '연월',
        '지역',
        '6대업종',
        '사업자수',
        '예측폐업수'
    ]
].head()

,연월,지역,6대업종,사업자수,예측폐업수
0,202606,중구,쇼핑업,2607,36.829162
1,202606,중구,식음료업,1946,25.954048
2,202606,서구,쇼핑업,1881,35.344704
3,202606,서구,식음료업,1393,25.934891
4,202606,동구,쇼핑업,3065,42.416344


In [13]:
future['추정_폐업강도_pct'] = (
    future['예측폐업수']
    / future['전월사업자수']
    * 100
)

In [14]:
future[
    [
        '연월',
        '지역',
        '6대업종',
        '사업자수',
        '전월사업자수',
        '예측폐업수',
        '추정_폐업강도_pct'
    ]
].to_csv(
    '/content/202606_부산_예측폐업.csv',
    index=False,
    encoding='utf-8-sig'
)

In [15]:
# 현재 모델을 보강하기 위해 21-25 부산 지역별 연간 폐업 수 데이터 대입
annual = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/폐업 및 신설/21-25전국_부산_폐업수.csv'
)

annual.head()

,연도,지역구분,지역명,폐업사업자_총계,법인,개인사업자,일반사업자,간이사업자,면세사업자
0,2025,전국,전국,975681,85270,890411,483210,313861,93340
1,2025,부산전체,부산,56651,4177,52474,28514,18942,5018
2,2025,부산구군,강서구,3850,408,3442,2310,834,298
3,2025,부산구군,금정구,3365,244,3121,1706,1127,288
4,2025,부산구군,기장군,3200,268,2932,1537,1079,316


In [16]:
# 다음 연도 예측용 변수로 변경
annual_context = annual[
    ['연도', '지역명', '폐업사업자_총계']
].copy()

# 2024년 폐업 → 2025년 예측변수
annual_context['연도'] = (
    annual_context['연도'] + 1
)

annual_context = annual_context.rename(
    columns={
        '지역명': '지역',
        '폐업사업자_총계': '전년도_폐업총계'
    }
)

In [17]:
# 기존 df와 병합

df['연도'] = df['date'].dt.year

df = df.merge(
    annual_context,
    on=['연도', '지역'],
    how='left'
)

In [18]:
numeric_features = [
    '사업자수',
    '사업자_증감수',
    '사업자_MoM',
    '사업자_YoY',

    '폐업_lag1',
    '폐업_lag2',
    '폐업_lag3',

    'month_sin',
    'month_cos',

    # 추가
    '전년도_폐업총계'
]

In [19]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_poisson_deviance
)

from scipy.stats import spearmanr

# 최종 모델 예측
pred_final = model.predict(X_test)

eval_df = test.copy()

eval_df['예측폐업수'] = pred_final
eval_df['Baseline예측'] = eval_df['폐업_lag1']

eval_df['모델절대오차'] = abs(
    eval_df['폐업수'] - eval_df['예측폐업수']
)

eval_df['Baseline절대오차'] = abs(
    eval_df['폐업수'] - eval_df['Baseline예측']
)


# -------------------------
# MAE
# -------------------------

model_mae = mean_absolute_error(
    eval_df['폐업수'],
    eval_df['예측폐업수']
)

baseline_mae = mean_absolute_error(
    eval_df['폐업수'],
    eval_df['Baseline예측']
)

improvement = (
    (baseline_mae - model_mae)
    / baseline_mae
    * 100
)


# -------------------------
# WAPE
# -------------------------

model_wape = (
    eval_df['모델절대오차'].sum()
    / eval_df['폐업수'].sum()
    * 100
)

baseline_wape = (
    eval_df['Baseline절대오차'].sum()
    / eval_df['폐업수'].sum()
    * 100
)


# -------------------------
# Spearman
# 실제 폐업 순위 vs 예측 순위
# -------------------------

rho, p_value = spearmanr(
    eval_df['폐업수'],
    eval_df['예측폐업수']
)


# -------------------------
# Poisson D²
# 평균만 예측하는 모델과 비교
# -------------------------

null_pred = np.repeat(
    y_train.mean(),
    len(eval_df)
)

d2 = 1 - (
    mean_poisson_deviance(
        eval_df['폐업수'],
        eval_df['예측폐업수']
    )
    /
    mean_poisson_deviance(
        eval_df['폐업수'],
        null_pred
    )
)


print("===== 최종 모델 평가 =====")
print(f"Model MAE : {model_mae:.3f}")
print(f"Baseline MAE : {baseline_mae:.3f}")
print(f"Baseline 대비 개선율 : {improvement:.2f}%")

print()

print(f"Model WAPE : {model_wape:.2f}%")
print(f"Baseline WAPE : {baseline_wape:.2f}%")

print()

print(f"Spearman rho : {rho:.3f}")
print(f"Spearman p-value : {p_value:.5f}")

print()

print(f"Poisson D² : {d2:.3f}")

===== 최종 모델 평가 =====
Model MAE : 8.953
Baseline MAE : 9.781
Baseline 대비 개선율 : 8.47%

Model WAPE : 17.14%
Baseline WAPE : 18.73%

Spearman rho : 0.937
Spearman p-value : 0.00000

Poisson D² : 0.863


In [20]:
import plotly.express as px

fig = px.scatter(
    eval_df,
    x='폐업수',
    y='예측폐업수',
    color='6대업종',
    hover_data=[
        '지역',
        '연월'
    ],
    title='실제 폐업수 vs 모델 예측 폐업수'
)

max_val = max(
    eval_df['폐업수'].max(),
    eval_df['예측폐업수'].max()
)

fig.add_shape(
    type='line',
    x0=0,
    y0=0,
    x1=max_val,
    y1=max_val,
    line=dict(
        dash='dash'
    )
)

fig.update_layout(
    xaxis_title='실제 폐업수',
    yaxis_title='예측 폐업수'
)

fig.show()

In [21]:
eval_df['잔차'] = (
    eval_df['폐업수']
    - eval_df['예측폐업수']
)

fig = px.scatter(
    eval_df,
    x='예측폐업수',
    y='잔차',
    color='6대업종',
    hover_data=[
        '지역',
        '연월'
    ],
    title='예측값에 따른 잔차 분포'
)

fig.add_hline(
    y=0,
    line_dash='dash'
)

fig.update_layout(
    xaxis_title='예측 폐업수',
    yaxis_title='잔차 (실제 - 예측)'
)

fig.show()

In [22]:
region_compare = eval_df.groupby('지역').agg(
    실제폐업평균=('폐업수', 'mean'),
    Model_MAE=('모델절대오차', 'mean'),
    Baseline_MAE=('Baseline절대오차', 'mean')
)

region_compare['개선율_pct'] = (
    (
        region_compare['Baseline_MAE']
        - region_compare['Model_MAE']
    )
    / region_compare['Baseline_MAE']
    * 100
)

region_compare = region_compare.sort_values(
    '개선율_pct',
    ascending=False
)

region_compare

,실제폐업평균,Model_MAE,Baseline_MAE,개선율_pct
지역,,,,
연제구,52.75,2.489661,12.50,80.082716
영도구,20.25,1.527620,4.50,66.052897
강서구,37.75,6.537176,14.00,53.305884
남구,59.00,6.437000,12.25,47.453062
사상구,46.00,4.238734,7.50,43.483549
북구,56.25,6.286476,10.00,37.135239
중구,22.25,8.785881,13.25,33.691462
사하구,64.75,11.133930,16.75,33.528775
금정구,48.25,9.710221,14.25,31.858098


In [23]:
fig = px.bar(
    region_compare.reset_index(),
    x='개선율_pct',
    y='지역',
    orientation='h',
    title='지역별 Baseline 대비 모델 개선율'
)

fig.add_vline(
    x=0,
    line_dash='dash'
)

fig.update_layout(
    xaxis_title='개선율 (%)',
    yaxis_title='지역'
)

fig.show()

In [24]:
from scipy.stats import wilcoxon

region_test = eval_df.groupby('지역').agg(
    Model_MAE=('모델절대오차', 'mean'),
    Baseline_MAE=('Baseline절대오차', 'mean')
)

stat, p = wilcoxon(
    region_test['Model_MAE'],
    region_test['Baseline_MAE'],
    alternative='less'
)

print("Wilcoxon statistic :", stat)
print("p-value :", p)

if p < 0.05:
    print("→ 모델의 오차가 Baseline보다 통계적으로 유의하게 작음")
else:
    print("→ Baseline보다 낮기는 하지만 통계적 유의성은 확인되지 않음")

Wilcoxon statistic : 46.0
p-value : 0.1372222900390625
→ Baseline보다 낮기는 하지만 통계적 유의성은 확인되지 않음


In [25]:
industry_map = {

    '쇼핑업': [
        '소매업'
    ],

    '여행/숙박업': [
        '숙박업',
        '여행알선 및 운수관련 서비스업'
    ],

    '운송업': [
        '운송업',
        '자동차관련 소매'
    ],

    '식음료업': [
        '음식점업'
    ],

    '의료웰니스업': [
        '보건업',
        '위생관련 서비스업'
    ],

    '서비스업': [
        '부동산업',
        '교육서비스업',
        '법무, 회계, 건축 및 상담업',
        '광고업 및 기타 산업관련 서비스업',
        '기타 서비스업',
        '인적용역',
        'IT관련 및 연구개발업',
        '기계장비 등 장비 임대업',
        '가사서비스업',
        '오락, 문화, 운동관련 산업 및 수리업'
    ]
}

In [26]:
import pandas as pd
import numpy as np

close_all = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/학습학습/부산_월별폐업_검증필터_학습용.csv',
    dtype={'연월': str}
)

biz = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/학습학습/2022-2026_사업자_현황_6대업종_통합.csv',
    dtype={'연월': str}
)

In [27]:
city_close = close_all[
    close_all['분석단위'] == '부산업종'
].copy()

In [28]:
def map_industry(x):

    for category, industries in industry_map.items():

        if x in industries:
            return category

    return np.nan


city_close['6대업종'] = (
    city_close['원업종']
    .apply(map_industry)
)

In [29]:
city_close = city_close.dropna(
    subset=['6대업종']
)

In [30]:
city_6 = (
    city_close
    .groupby(
        ['연월', '6대업종'],
        as_index=False
    )
    ['폐업수']
    .sum()
    .rename(
        columns={
            '폐업수': '부산_업종폐업수'
        }
    )
)

city_6.head()

,연월,6대업종,부산_업종폐업수
0,202410,서비스업,984
1,202410,쇼핑업,972
2,202410,식음료업,650
3,202410,여행/숙박업,66
4,202410,운송업,221


In [31]:
biz_region = biz[
    biz['지역'] != '부산광역시 합계'
].copy()

biz_region['사업자수'] = pd.to_numeric(
    biz_region['당월①'],
    errors='coerce'
)

biz_region[
    '부산_해당업종_사업자수'
] = (
    biz_region
    .groupby(
        ['연월', '업종']
    )['사업자수']
    .transform('sum')
)

In [32]:
biz_region['지역사업자비중'] = (
    biz_region['사업자수']
    /
    biz_region['부산_해당업종_사업자수']
)

In [33]:
proxy = biz_region.merge(
    city_6,
    left_on=[
        '연월',
        '업종'
    ],
    right_on=[
        '연월',
        '6대업종'
    ],
    how='inner'
)

In [34]:
proxy['추정폐업수'] = (
    proxy['부산_업종폐업수']
    *
    proxy['지역사업자비중']
)

In [35]:
actual = close_all[
    (close_all['분석단위'] == '지역업종')
    &
    (
        close_all['원업종']
        .isin([
            '소매업',
            '음식점업'
        ])
    )
].copy()

In [36]:
actual['6대업종'] = (
    actual['원업종']
    .map({
        '소매업': '쇼핑업',
        '음식점업': '식음료업'
    })
)

actual = actual[
    [
        '연월',
        '지역',
        '6대업종',
        '폐업수'
    ]
]

In [37]:
six_df = proxy.merge(
    actual,
    on=[
        '연월',
        '지역',
        '6대업종'
    ],
    how='left'
)

In [38]:
six_df['학습폐업수'] = np.where(
    six_df['폐업수'].notna(),
    six_df['폐업수'],
    six_df['추정폐업수']
)

In [39]:
six_df['데이터구분'] = np.where(
    six_df['폐업수'].notna(),
    '실측',
    '약한라벨'
)

In [40]:
six_df['sample_weight'] = np.where(
    six_df['데이터구분'] == '실측',
    1.0,
    0.35
)

In [41]:
print(df.columns.tolist())
print(train.columns.tolist())

['연월', '지역', '원업종', '6대업종', '폐업수', '6대업종매핑신뢰도', '지역합계검증', '원천발행연월', '원천값유형', '중복검증원천수', '업종', '시도', '당월①', '전월②', '증감율(①/②)', '전년동월③', '증감율(①/③)', '사업자수', '전월사업자수', '사업자_증감수', '사업자_MoM', '사업자_YoY', 'date', '폐업_lag1', '폐업_lag2', '폐업_lag3', 'month_sin', 'month_cos', '연도', '전년도_폐업총계']
['연월', '지역', '원업종', '6대업종', '폐업수', '6대업종매핑신뢰도', '지역합계검증', '원천발행연월', '원천값유형', '중복검증원천수', '업종', '시도', '당월①', '전월②', '증감율(①/②)', '전년동월③', '증감율(①/③)', '사업자수', '전월사업자수', '사업자_증감수', '사업자_MoM', '사업자_YoY', 'date', '폐업_lag1', '폐업_lag2', '폐업_lag3', 'month_sin', 'month_cos']


In [42]:
df['연도'] = df['date'].dt.year

In [43]:
annual = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/폐업 및 신설/21-25전국_부산_폐업수.csv'
)

print(annual.columns)
annual.head()

Index(['연도', '지역구분', '지역명', '폐업사업자_총계', '법인', '개인사업자', '일반사업자', '간이사업자',
       '면세사업자'],
      dtype='object')


,연도,지역구분,지역명,폐업사업자_총계,법인,개인사업자,일반사업자,간이사업자,면세사업자
0,2025,전국,전국,975681,85270,890411,483210,313861,93340
1,2025,부산전체,부산,56651,4177,52474,28514,18942,5018
2,2025,부산구군,강서구,3850,408,3442,2310,834,298
3,2025,부산구군,금정구,3365,244,3121,1706,1127,288
4,2025,부산구군,기장군,3200,268,2932,1537,1079,316


In [44]:
annual_context = annual[
    ['연도', '지역명', '폐업사업자_총계']
].copy()

# 예:
# 2024년 폐업총계 → 2025년 예측 설명변수
annual_context['연도'] = annual_context['연도'] + 1

annual_context = annual_context.rename(
    columns={
        '지역명': '지역',
        '폐업사업자_총계': '전년도_폐업총계'
    }
)

In [45]:
df = df.merge(
    annual_context,
    on=['연도', '지역'],
    how='left'
)

In [46]:
[col for col in df.columns if '폐업총계' in col]

['전년도_폐업총계_x', '전년도_폐업총계_y']

In [47]:
# 기존에 만들어진 전년도 폐업총계 관련 컬럼 모두 제거
drop_cols = [
    col for col in df.columns
    if '전년도_폐업총계' in col
]

df = df.drop(columns=drop_cols, errors='ignore')

print("삭제 후 :", [
    col for col in df.columns
    if '폐업총계' in col
])

삭제 후 : []


In [48]:
df['연도'] = df['date'].dt.year

In [49]:
annual = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/폐업 및 신설/21-25전국_부산_폐업수.csv'
)

print(annual.columns.tolist())

['연도', '지역구분', '지역명', '폐업사업자_총계', '법인', '개인사업자', '일반사업자', '간이사업자', '면세사업자']


In [50]:
annual_context = annual[
    ['연도', '지역명', '폐업사업자_총계']
].copy()

# 전년도 폐업수를 다음 연도 설명변수로 사용
annual_context['연도'] = (
    annual_context['연도'] + 1
)

annual_context = annual_context.rename(
    columns={
        '지역명': '지역',
        '폐업사업자_총계': '전년도_폐업총계'
    }
)

annual_context.head()

,연도,지역,전년도_폐업총계
0,2026,전국,975681
1,2026,부산,56651
2,2026,강서구,3850
3,2026,금정구,3365
4,2026,기장군,3200


In [51]:
df = df.merge(
    annual_context,
    on=['연도', '지역'],
    how='left'
)

In [52]:
print([
    col for col in df.columns
    if '폐업총계' in col
])

['전년도_폐업총계']


In [53]:
df[
    ['연월', '지역', '연도', '전년도_폐업총계']
].head(20)

,연월,지역,연도,전년도_폐업총계
0,202410,강서구,2024,3755
1,202410,강서구,2024,3755
2,202410,금정구,2024,3645
3,202410,금정구,2024,3645
4,202410,기장군,2024,3429
5,202410,기장군,2024,3429
6,202410,남구,2024,4104
7,202410,남구,2024,4104
8,202410,동구,2024,2212
9,202410,동구,2024,2212


In [54]:
# 1. 기존 df에 연도 생성
df['연도'] = df['date'].dt.year

# 2. 기존에 꼬인 폐업총계 컬럼 제거
df = df.drop(
    columns=[
        c for c in df.columns
        if '전년도_폐업총계' in c
    ],
    errors='ignore'
)

# 3. 연간 폐업 데이터 불러오기
annual = pd.read_csv(
    '/content/drive/MyDrive/MBCA/1차 팀 프로젝트/데이터 파일/폐업 및 신설/21-25전국_부산_폐업수.csv'
)

# 4. 전년도 폐업총계 만들기
annual_context = annual[
    ['연도', '지역명', '폐업사업자_총계']
].copy()

# 2024년 폐업값 → 2025년 설명변수
annual_context['연도'] = (
    annual_context['연도'] + 1
)

annual_context = annual_context.rename(
    columns={
        '지역명': '지역',
        '폐업사업자_총계': '전년도_폐업총계'
    }
)

# 5. df에 다시 merge
df = df.merge(
    annual_context,
    on=['연도', '지역'],
    how='left'
)

# -----------------------------
# ★ 여기서 반드시 확인
# -----------------------------
print("df 컬럼 존재 여부:",
      '전년도_폐업총계' in df.columns)

print(
    df[
        ['연월', '지역', '연도', '전년도_폐업총계']
    ].head()
)

# 6. model_df를 다시 생성
model_df = df.dropna(
    subset=[
        '폐업_lag1',
        '폐업_lag2',
        '폐업_lag3',
        '사업자수',
        '사업자_MoM',
        '사업자_YoY',
        '전년도_폐업총계'
    ]
).copy()

# 7. train / test를 반드시 다시 생성
train = model_df[
    model_df['date'] <= '2026-03-01'
].copy()

test = model_df[
    model_df['date'].between(
        '2026-04-01',
        '2026-05-01'
    )
].copy()

# -----------------------------
# ★ train에도 있는지 확인
# -----------------------------
print(
    "train 컬럼 존재 여부:",
    '전년도_폐업총계' in train.columns
)

print(
    "test 컬럼 존재 여부:",
    '전년도_폐업총계' in test.columns
)

print("train shape :", train.shape)
print("test shape :", test.shape)

df 컬럼 존재 여부: True
       연월   지역    연도  전년도_폐업총계
0  202410  강서구  2024      3755
1  202410  강서구  2024      3755
2  202410  금정구  2024      3645
3  202410  금정구  2024      3645
4  202410  기장군  2024      3429
train 컬럼 존재 여부: True
test 컬럼 존재 여부: True
train shape : (288, 30)
test shape : (64, 30)


In [55]:
numeric_features = [
    '사업자수',
    '사업자_증감수',
    '사업자_MoM',
    '사업자_YoY',
    '폐업_lag1',
    '폐업_lag2',
    '폐업_lag3',
    'month_sin',
    'month_cos',
    '전년도_폐업총계'
]

X_train = train[
    categorical_features + numeric_features
]

y_train = train['폐업수']

X_test = test[
    categorical_features + numeric_features
]

y_test = test['폐업수']

print(X_train.shape)
print(X_train.columns.tolist())

(288, 12)
['지역', '6대업종', '사업자수', '사업자_증감수', '사업자_MoM', '사업자_YoY', '폐업_lag1', '폐업_lag2', '폐업_lag3', 'month_sin', 'month_cos', '전년도_폐업총계']


In [57]:
print("train 컬럼:")
print(train.columns.tolist())

print()

print("폐업 관련 컬럼:")
print([
    c for c in train.columns
    if '폐업' in c
])

train 컬럼:
['연월', '지역', '원업종', '6대업종', '폐업수', '6대업종매핑신뢰도', '지역합계검증', '원천발행연월', '원천값유형', '중복검증원천수', '업종', '시도', '당월①', '전월②', '증감율(①/②)', '전년동월③', '증감율(①/③)', '사업자수', '전월사업자수', '사업자_증감수', '사업자_MoM', '사업자_YoY', 'date', '폐업_lag1', '폐업_lag2', '폐업_lag3', 'month_sin', 'month_cos', '연도', '전년도_폐업총계']

폐업 관련 컬럼:
['폐업수', '폐업_lag1', '폐업_lag2', '폐업_lag3', '전년도_폐업총계']


In [58]:
y_train = train['폐업수']
y_test = test['폐업수']

In [59]:
model.fit(
    X_train,
    y_train
)

pred = model.predict(X_test)

In [60]:
from sklearn.metrics import mean_absolute_error

model_mae = mean_absolute_error(
    y_test,
    pred
)

baseline_mae = mean_absolute_error(
    y_test,
    test['폐업_lag1']
)

print("최종 Poisson MAE :", model_mae)
print("Baseline MAE :", baseline_mae)

print(
    "개선율 :",
    (baseline_mae - model_mae)
    / baseline_mae * 100,
    "%"
)

최종 Poisson MAE : 8.952592143223491
Baseline MAE : 9.78125
개선율 : 8.47190141113364 %


In [61]:
print(numeric_features)

['사업자수', '사업자_증감수', '사업자_MoM', '사업자_YoY', '폐업_lag1', '폐업_lag2', '폐업_lag3', 'month_sin', 'month_cos', '전년도_폐업총계']


In [62]:
print(X_train.columns.tolist())

['지역', '6대업종', '사업자수', '사업자_증감수', '사업자_MoM', '사업자_YoY', '폐업_lag1', '폐업_lag2', '폐업_lag3', 'month_sin', 'month_cos', '전년도_폐업총계']


In [63]:
print(
    X_train[
        ['전년도_폐업총계']
    ].describe()
)

          전년도_폐업총계
count   288.000000
mean   3645.395833
std    1604.718071
min    1374.000000
25%    2969.250000
50%    3609.000000
75%    4046.000000
max    7315.000000


In [64]:
model.fit(
    X_train,
    y_train
)

pred = model.predict(
    X_test
)

model_mae = mean_absolute_error(
    y_test,
    pred
)

baseline_mae = mean_absolute_error(
    y_test,
    test['폐업_lag1']
)

print("Poisson MAE :", model_mae)
print("Baseline MAE :", baseline_mae)

Poisson MAE : 8.952592143223491
Baseline MAE : 9.78125


In [65]:
from scipy.stats import spearmanr

rho, p = spearmanr(
    y_test,
    pred
)

print("Spearman rho :", rho)
print("p-value :", p)

Spearman rho : 0.9374721233742085
p-value : 4.10854069703765e-30
